In [1]:
import polars as pl
import datetime as dt
from sklearn.model_selection import TimeSeriesSplit
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline


EDA

Annual, monthly, weekly sales analysis

In [3]:
# 1. Load Raw Data
train_raw = pd.read_csv('store-sales-time-series-forecasting/train.csv')
test_raw = pd.read_csv('store-sales-time-series-forecasting/test.csv') # The Kaggle submission set

oil_raw = pd.read_csv('store-sales-time-series-forecasting/oil.csv')
stores_raw = pd.read_csv('store-sales-time-series-forecasting/stores.csv')
holidays_raw = pd.read_csv('store-sales-time-series-forecasting/holidays_events.csv')


In [4]:
# analysis of holidays

unique_holiday_type = holidays_raw['type'].unique()
print("Unique Holiday Types")
print(unique_holiday_type)

unique_holiday_locale = holidays_raw['locale'].unique()
print("Unique Holiday locale")
print(unique_holiday_locale)


Unique Holiday Types
<StringArray>
['Holiday', 'Transfer', 'Additional', 'Bridge', 'Work Day', 'Event']
Length: 6, dtype: str
Unique Holiday locale
<StringArray>
['Local', 'Regional', 'National']
Length: 3, dtype: str


In [19]:
# create custom preprocessor as standard scikit-learn pipelines don't natively handle relational
# database operations like pd.merge() or temporal operations like reindexing calendars.


class StoreSalesPreprocessor(BaseEstimator, TransformerMixin):
    """
    Custom Scikit-Learn Transformer for the Kaggle Store Sales Competition.
    Handles dates, oil interpolation, and relational table merges.
    """
    def __init__(self,
 oil_df, stores_df, holidays_df=None):
        # Pass the auxiliary dataframes when initializing the pipeline
        self.oil_df = oil_df.copy()
        self.stores_df = stores_df.copy()
        self.holidays_df = holidays_df.copy() if holidays_df is not None else None
        
    def fit(self, X, y=None):
        """
        The fit method learns parameters from the training data.
        We process the auxiliary tables here so it only happens once.
        """
        print("Fitting pipeline: Processing auxiliary tables...")
        
        # 1. Process Oil Data (The logic we discussed earlier)
        self.processed_oil_ = self._process_oil(self.oil_df)
        
        # 2. Process Stores Data (Any aggregations or cleaning on stores happens here)
        # For example, mapping 'city' or 'state' to broader regions if you wanted
        # self.processed_stores_ = self.stores_df.copy()
        self.processed_stores_ = self._process_stores(self.stores_df)

        self.processed_holidays_ = self._process_holidays(self.holidays_df)
        
        # 3. (Optional) Target Encoding Aggregations could go here
        # E.g., learning the historical mean sales per store from X and y
        
        return self

    def transform(self, X):
        """
        The transform method applies the merges to the Train or Test data.
        """
        print(f"Transforming data ({len(X)} rows)...")
        X_out = X.copy()
        
        # 1. Ensure date is a datetime object
        X_out['date'] = pd.to_datetime(X_out['date'])
        
        # 2. Merge Oil
        X_out = pd.merge(X_out, self.processed_oil_, on='date', how='left')
        
        # 3. Merge Stores
        X_out = pd.merge(X_out, self.processed_stores_, on='store_nbr', how='left')
        
        # 3. Merge Holidays
        X_out = pd.merge(X_out, self.processed_holidays_, on='date', how='left')
        
        # 4. Extract base calendar features (always good to have in the pipeline)
        X_out['day_of_week'] = X_out['date'].dt.dayofweek
        X_out['month'] = X_out['date'].dt.month
        X_out['year'] = X_out['date'].dt.year
        X_out['is_weekend'] = X_out['day_of_week'].isin([5, 6]).astype(int)
        
        return X_out

    def _process_oil(self, oil):
        """Helper method to encapsulate the oil interpolation logic"""
        oil['date'] = pd.to_datetime(oil['date'])
        
        # Create continuous calendar bounds based on the oil dataset
        calendar = pd.date_range(start=oil['date'].min(), end=oil['date'].max())
        
        # Reindex and Interpolate
        oil_continuous = oil.set_index('date').reindex(calendar).rename_axis('date').reset_index()
        
        # forward fill, then backward fill for the very first missing day (Jan 1, 2013)
        oil_continuous['dcoilwtico'] = oil_continuous['dcoilwtico'].ffill().bfill()
        
        # Rename column for clarity
        oil_continuous = oil_continuous.rename(columns={'dcoilwtico': 'oil_price'})
        return oil_continuous
    
    def _process_holidays(self, holidays):
        """Helper method to transform holidays so it can be incorporated with other tables"""
        holidays['date'] = pd.to_datetime(holidays['date'])

        holidays = holidays.rename(columns={'locale': 'holiday_location', 'locale_name': 'holiday_location_name', 'type': 'holiday_type', 'description': 'holiday_description',  'transferred': 'holiday_transferred'})

        return holidays
    
    def _process_stores(self, stores):
        """Helper method to transform stores so it can be incorporated with other tables"""

        stores = stores.rename(columns={'city': 'store_city', 'state': 'store_state', 'type': 'store_type', 'cluster': 'store_cluster'})

        return stores



In [20]:
# 2. Initialize your custom transformer
# Notice we pass the auxiliary tables into the initialization
feature_pipeline = Pipeline([
    ('preprocessor', StoreSalesPreprocessor(oil_df=oil_raw, stores_df=stores_raw, holidays_df=holidays_raw))
    # You can add StandardScalers or XGBoost models here later!
])

# 3. Fit and Transform the Training Data
# .fit() processes oil and stores. .transform() merges them into train.
X_train_processed = feature_pipeline.fit_transform(train_raw)

# 4. Transform the Test Data
# Reuses the exact same oil and store tables, ensuring 0 leakage and perfect consistency
X_test_processed = feature_pipeline.transform(test_raw)

# Look at the result
display(X_train_processed.head(100))


Fitting pipeline: Processing auxiliary tables...
Transforming data (3000888 rows)...
Transforming data (28512 rows)...


,id,date,store_nbr,family,sales,onpromotion,oil_price,store_city,store_state,store_type,store_cluster,holiday_type,holiday_location,holiday_location_name,holiday_description,holiday_transferred,day_of_week,month,year,is_weekend
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,93.14,Quito,Pichincha,D,13,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0
1,1,2013-01-01,1,BABY CARE,0.0,0,93.14,Quito,Pichincha,D,13,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0
2,2,2013-01-01,1,BEAUTY,0.0,0,93.14,Quito,Pichincha,D,13,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0
3,3,2013-01-01,1,BEVERAGES,0.0,0,93.14,Quito,Pichincha,D,13,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0
4,4,2013-01-01,1,BOOKS,0.0,0,93.14,Quito,Pichincha,D,13,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,2013-01-01,11,PREPARED FOODS,0.0,0,93.14,Cayambe,Pichincha,B,6,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0
96,96,2013-01-01,11,PRODUCE,0.0,0,93.14,Cayambe,Pichincha,B,6,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0
97,97,2013-01-01,11,SCHOOL AND OFFICE SUPPLIES,0.0,0,93.14,Cayambe,Pichincha,B,6,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0
98,98,2013-01-01,11,SEAFOOD,0.0,0,93.14,Cayambe,Pichincha,B,6,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0


In [21]:
display(X_train_processed[10000:10010])


,id,date,store_nbr,family,sales,onpromotion,oil_price,store_city,store_state,store_type,store_cluster,holiday_type,holiday_location,holiday_location_name,holiday_description,holiday_transferred,day_of_week,month,year,is_weekend
10000,10000,2013-01-06,4,BABY CARE,0.000,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10001,10001,2013-01-06,4,BEAUTY,5.000,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10002,10002,2013-01-06,4,BEVERAGES,1869.000,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10003,10003,2013-01-06,4,BOOKS,0.000,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10004,10004,2013-01-06,4,BREAD/BAKERY,518.348,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10005,10005,2013-01-06,4,CELEBRATION,0.000,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10006,10006,2013-01-06,4,CLEANING,1492.000,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10007,10007,2013-01-06,4,DAIRY,616.000,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10008,10008,2013-01-06,4,DELI,317.962,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10009,10009,2013-01-06,4,EGGS,325.000,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1


EDA braindump

Sales

Time Statistics

Yearly average sales
Monthly sales (year on year)
Weekly sales (year on year)
Daily sales (year on year)

Store Statistics

Sales by Store Type (over the years)
Sales by 

Sales by family (of item)

Autocorrelation, partial autocorrelation


Feature Engineering

For future - way to join holiday location with store location 


Calendar Features: Extract day_of_week, day_of_month, month, year, is_weekend, is_payday (e.g., 15th and last day of the month are huge for retail).


Lag Features: Sales from exactly 1 week ago ($t-7$), 2 weeks ago ($t-14$), etc.


Rolling Window Statistics: Moving averages, rolling standard deviations, rolling min/max over the last 7, 14, or 28 days.


Target Encoding: Average sales per store, average sales per product family.


Event Proximity: Days until the next holiday, days since the last holiday.
